# 1.1 统一视角：用代码重现 E[R] = βλ + α

对应原书第1章 1.1 节。本 notebook 的目标：

1. 用教学小样本（4 只股票 × 2 个因子）计算预期收益 E[R] = βλ；
2. 加入 α 制造一个「异象」，并在图上看出定价误差；
3. 模拟因子收益率时序，验证协方差分解 Σ = βΣλβ′ + Σε。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 数据目录：兼容从仓库根目录或 chapters/xx/notebooks 目录启动
from pathlib import Path
def _find_sample():
    for p in [Path('data/sample'), Path.cwd().parents[2] / 'data' / 'sample']:
        if p.exists():
            return p
    raise FileNotFoundError('找不到 data/sample，请从仓库根目录启动 notebook')
SAMPLE = _find_sample()
plt.rcParams['figure.dpi'] = 110

## 第 0 步：加载教学小样本

数据是原书表 2.1 的结构：4 只股票在因子 A、B 上的暴露（beta）与各自的特质性风险（idio risk）。

In [ ]:
teach = pd.read_csv(SAMPLE / 'teaching_sample.csv')
teach

## 第 1 步：预期收益 = 暴露 × 因子溢价

设因子溢价（月均）：λ_A = 0.8%，λ_B = 0.4%。
**预期收益 = 暴露矩阵 β 与溢价向量 λ 的矩阵乘法。**

In [ ]:
lam = np.array([0.008, 0.004])  # 因子溢价 λ（月均）
beta = teach[['factor_a_exposure', 'factor_b_exposure']].to_numpy()
er_model = beta @ lam  # 多因子模型隐含的预期收益
pd.Series(er_model, index=teach['asset'], name='E[R]_model').round(4)

## 第 2 步：加入 α —— 制造一个「异象」

给 stock_2 一个显著为正的定价误差 α=1.2%/月，它就成了多因子模型解释不了的「异象」：
**同样的暴露，更高的预期收益。**

In [ ]:
alpha = np.array([0.0, 0.012, 0.0, 0.0])  # 只有 stock_2 有 α
er_actual = er_model + alpha
out = pd.DataFrame({'asset': teach['asset'], 'E[R]_model': er_model,
                    'alpha': alpha, 'E[R]_actual': er_actual})
out.round(4)

## 第 3 步：画出来 —— 定价误差看得见

横轴：因子 A 上的暴露；纵轴：预期收益。模型隐含的关系是过原点的直线（斜率 λ_A）；
**点到直线的垂直距离就是 α** —— 这就是书中图 2.3 的截面视角。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
xs = np.linspace(0.6, 1.3, 10)
ax.plot(xs, xs * lam[0], 'k--', label='E[R] = β·λ（模型直线）')
for i, row in out.iterrows():
    b = beta[i, 0]
    marker = 'o' if row['alpha'] == 0 else '^'
    ax.scatter(b, row['E[R]_actual'], s=120, marker=marker,
               label=f"{row['asset']}" + ('（异象 ⭐）' if row['alpha'] != 0 else ''))
    if row['alpha'] != 0:
        ax.vlines(b, row['E[R]_model'], row['E[R]_actual'], color='red', linewidth=2)
        ax.annotate('α = 1.2%/月', xy=(b, row['E[R]_model']),
                    xytext=(b + 0.02, row['E[R]_model'] + 0.001), color='red')
ax.set_xlabel('因子 A 暴露 β_A')
ax.set_ylabel('预期收益 E[R]（月均）')
ax.set_title('统一视角：E[R] = βλ + α')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 第 4 步：时序模拟 —— 协方差分解 Σ = βΣλβ′ + Σε

从截面切到时序：模拟 T=120 个月的因子收益与个股收益，验证：
**用因子算出的协方差结构 ≈ 真实的协方差矩阵**（降维的根据）。

In [ ]:
rng = np.random.default_rng(42)
T = 120
cov_lam = np.array([[0.0009, 0.0001], [0.0001, 0.0004]])  # 因子协方差
lam_t = rng.multivariate_normal(lam, cov_lam, size=T)     # 因子收益时序
# 特质噪声：形状 (T, 4) 生成后转置为 (4, T)，与 alpha/beta@lam_t.T 对齐
idio = rng.normal(0, teach['idio_risk'].to_numpy(), size=(T, 4)).T
ret = alpha.reshape(-1, 1) + beta @ lam_t.T + idio  # 个股收益时序（式 1.5 的数值版）
print('个股收益的样本协方差 Σ:')
print(np.cov(ret).round(5))

In [ ]:
# 用因子重建协方差：Σ_hat = βΣλβ' + Σε
sigma_eps = np.diag(teach['idio_risk'].to_numpy() ** 2)
sigma_hat = beta @ cov_lam @ beta.T + sigma_eps
sigma_true = np.cov(ret)
err = np.abs(sigma_hat - sigma_true).max()
print(f'重建协方差 vs 真实协方差的最大绝对误差: {err:.5f}')
print('=> 协方差分解 Σ = βΣλβ′ + Σε 成立，这就是「因子=降维」的数学依据')

## 小结

- 预期收益是**截面**概念：E[R] = βλ，暴露高 → 预期收益高；
- **α ≠ 0** 的资产 = 异象（多因子模型解释不了的部分）；
- 协方差是**时序**概念：Σ = βΣλβ′ + Σε，因子把 N×N 的协方差问题降成 K×K；
- 回到第1章检查站：现在你能用自己的话说清这三个概念了吗？